[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-06-rag-search.ipynb#scrollTo=fa01ab02)

---
# Day 6 · RAG & Agentic Search
**certified-journeys / claude-api-certified** · Embeddings, chunking & retrieval pipelines

> **Goal for today:** Build a full RAG pipeline from scratch — chunk documents, generate embeddings, search by similarity, and inject retrieved context into Claude prompts to ground answers in your own data.

In [ ]:
%pip install -q anthropic sentence-transformers numpy

In [ ]:
import os
import json
import numpy as np
import anthropic
from sentence_transformers import SentenceTransformer

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

# Load a lightweight embedding model (runs locally, no API key needed)
# Production equivalent: Voyage AI (voyage-3) via the Anthropic API
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

## Step 1 · Chunk documents

Chunking splits long documents into segments that fit within embedding model limits and retrieval windows.

| Strategy | Best for |
|---|---|
| Fixed-size (chars/words) | Simple, predictable, fast |
| Sentence boundary | Preserves semantic coherence |
| Sliding window (overlap) | Reduces context loss at chunk boundaries |
| Semantic chunking | Groups by topic — best quality, slowest |

In [ ]:
# Synthetic knowledge base — Anthropic API facts
DOCUMENTS = [
    """The Anthropic Messages API is the primary interface for interacting with Claude models.
It accepts a list of messages alternating between user and assistant roles. Each message
has a role and content field. The content can be a string or a list of content blocks.
Supported content block types include text, image, document, and tool_use.""",

    """Prompt caching in the Claude API allows developers to cache large portions of the prompt
context to reduce latency and cost on repeated requests. To enable caching, add a cache_control
parameter with type 'ephemeral' to content blocks. Cached content must be at least 1024 tokens.
Cache hits reduce input token costs by approximately 90% and are available for up to 5 minutes.""",

    """Claude's tool use feature allows the model to call developer-defined functions. When Claude
wants to use a tool, the stop_reason is 'tool_use' and the response includes ToolUseBlock objects.
Each ToolUseBlock contains a name, input dict, and unique id. Developers must execute the tool
and return results as tool_result content blocks referencing the same tool_use_id.""",

    """Extended thinking gives Claude extra time to reason through complex problems before
responding. It is available on claude-sonnet-4-5 and above. Enable it by passing
thinking={'type': 'enabled', 'budget_tokens': N} to messages.create(). The response includes
a ThinkingBlock with the internal reasoning. Extended thinking uses additional output tokens.""",

    """The Model Context Protocol (MCP) is an open standard for connecting AI models to external
data sources and tools. An MCP server exposes tools, resources, and prompts. An MCP client
connects to servers and makes capabilities available to AI models. Anthropic developed MCP
and Claude Code uses it extensively for file system access and tool integration.""",
]

def chunk_fixed(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """Split text into fixed-size word chunks with overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


# Chunk all documents
all_chunks = []
for doc_id, doc in enumerate(DOCUMENTS):
    for chunk in chunk_fixed(doc, chunk_size=60, overlap=15):
        all_chunks.append({"doc_id": doc_id, "text": chunk})

print(f"Total chunks: {len(all_chunks)}")
print(f"Sample chunk: {all_chunks[0]['text'][:100]}...")

**What just happened?**
- Fixed-size chunking with overlap (15 words) prevents context loss at chunk boundaries.
- **Chunk size is your most important RAG hyperparameter** — too small = no context; too large = retrieval noise.
- We keep `doc_id` so we can trace retrieved chunks back to their source document.
- For production: preserve document metadata (title, URL, section) for citation and filtering.

## Step 2 · Generate embeddings and build an index

Embeddings are dense vector representations of text. Similar texts have similar vectors, enabling semantic search.

In [ ]:
class VectorStore:
    """Simple in-memory vector store with cosine similarity search."""

    def __init__(self, model: SentenceTransformer):
        self.model = model
        self.chunks: list[dict] = []
        self.embeddings: np.ndarray | None = None

    def add(self, chunks: list[dict]) -> None:
        """Embed and index a list of chunk dicts with a 'text' key."""
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        print(f"Embedding {len(texts)} chunks...")
        self.embeddings = self.model.encode(texts, normalize_embeddings=True)
        print(f"Index ready. Shape: {self.embeddings.shape}")

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        """Return top_k most similar chunks for a query."""
        query_vec = self.model.encode([query], normalize_embeddings=True)
        # Cosine similarity (vectors are normalized so dot product == cosine sim)
        scores = (self.embeddings @ query_vec.T).squeeze()
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            {**self.chunks[i], "score": float(scores[i])}
            for i in top_indices
        ]


store = VectorStore(embedder)
store.add(all_chunks)

# Test retrieval
results = store.search("How does prompt caching work?", top_k=3)
for r in results:
    print(f"Score: {r['score']:.3f} | {r['text'][:80]}...")

**What just happened?**
- `normalize_embeddings=True` makes dot product equivalent to cosine similarity — no extra computation needed.
- **`np.argsort(scores)[::-1][:top_k]`** — sort descending, take top k indices.
- Scores range from 0 to 1. Scores above 0.7 are generally relevant; below 0.4 are likely noise.
- For production: replace `VectorStore` with ChromaDB, LanceDB, or pgvector for persistence and scale.

## Step 3 · Build the RAG prompt and call Claude

Inject retrieved chunks as context in the prompt, then ask Claude to answer from that context only.

In [ ]:
RAG_SYSTEM = """You are a helpful assistant with access to a knowledge base about the Anthropic Claude API.
Answer questions using ONLY the provided context. If the context doesn't contain the answer, say so.
Be concise and accurate. Always indicate which part of the context supports your answer."""

def rag_answer(question: str, top_k: int = 3) -> dict:
    """Retrieve relevant chunks and use them to answer a question."""
    # 1. Retrieve
    retrieved = store.search(question, top_k=top_k)

    # 2. Build context string
    context_parts = []
    for i, chunk in enumerate(retrieved):
        context_parts.append(f"[{i+1}] (score: {chunk['score']:.2f}) {chunk['text']}")
    context = "\n\n".join(context_parts)

    # 3. Build prompt with XML structure
    prompt = f"""<context>
{context}
</context>

<question>
{question}
</question>"""

    # 4. Call Claude
    r = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=RAG_SYSTEM,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "question": question,
        "answer": r.content[0].text,
        "retrieved_chunks": len(retrieved),
        "top_score": retrieved[0]["score"] if retrieved else 0
    }


# Test questions
for q in [
    "What is prompt caching and how much does it cost?",
    "How do tool_use and tool_result relate to each other?",
    "What is MCP and who created it?"
]:
    result = rag_answer(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer'][:200]}")
    print(f"Top retrieval score: {result['top_score']:.3f}\n")

**What just happened?**
- The numbered context chunks `[1], [2], [3]` let Claude cite sources in its answer.
- **Score thresholding:** if `top_score < 0.4`, consider returning "I don't have information on that" instead of hallucinating.
- `system` prompt constrains Claude to only use the context — reduces hallucination risk significantly.
- **Retrieval quality > generation quality** in RAG systems. A bad retrieval step defeats the best prompt.

## Step 4 · Measure retrieval quality with precision@k

Precision@k measures what fraction of the top k retrieved results are actually relevant. Build a small labeled test set to track retrieval quality.

In [ ]:
# Labeled evaluation set: query → which doc_ids should be retrieved
RETRIEVAL_EVAL = [
    {"query": "prompt caching cost and duration", "relevant_docs": {1}},
    {"query": "tool_use stop reason and ToolUseBlock", "relevant_docs": {2}},
    {"query": "extended thinking budget tokens", "relevant_docs": {3}},
    {"query": "MCP server resources prompts tools", "relevant_docs": {4}},
    {"query": "Messages API content blocks types", "relevant_docs": {0}},
]

def precision_at_k(retrieved: list[dict], relevant_docs: set[int], k: int = 3) -> float:
    """Fraction of top-k retrieved chunks that come from a relevant document."""
    top_k = retrieved[:k]
    hits = sum(1 for r in top_k if r["doc_id"] in relevant_docs)
    return hits / k


scores = []
for example in RETRIEVAL_EVAL:
    retrieved = store.search(example["query"], top_k=3)
    p_at_3 = precision_at_k(retrieved, example["relevant_docs"], k=3)
    scores.append(p_at_3)
    print(f"Query: {example['query'][:50]}")
    print(f"  P@3: {p_at_3:.2f} | Top doc_ids: {[r['doc_id'] for r in retrieved[:3]]}")

print(f"\nMean P@3: {sum(scores)/len(scores):.2f}")

**What just happened?**
- Precision@3 tells you: of the 3 chunks Claude sees, how many are from the right document?
- **A P@3 below 0.5 means retrieval is noisy** — optimize chunk size, embedding model, or both before tuning the prompt.
- Always build a labeled retrieval eval set before deploying a RAG system to production.
- Recall@k (did we retrieve any relevant chunk?) is also useful — compute both for full picture.

## Step 5 · Handle the no-answer case

RAG systems must gracefully handle queries that have no relevant context in the knowledge base.

In [ ]:
def rag_answer_safe(question: str, top_k: int = 3, min_score: float = 0.35) -> dict:
    """RAG with score threshold — falls back gracefully when context is poor."""
    retrieved = store.search(question, top_k=top_k)
    top_score = retrieved[0]["score"] if retrieved else 0

    if top_score < min_score:
        return {
            "question": question,
            "answer": "I don't have information about this in my knowledge base.",
            "fallback": True,
            "top_score": top_score
        }

    context = "\n\n".join([f"[{i+1}] {c['text']}" for i, c in enumerate(retrieved)])
    prompt = f"<context>\n{context}\n</context>\n\n<question>\n{question}\n</question>"

    r = client.messages.create(
        model=MODEL, max_tokens=256, system=RAG_SYSTEM,
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "question": question,
        "answer": r.content[0].text,
        "fallback": False,
        "top_score": top_score
    }


# Test with an out-of-scope question
out_of_scope = rag_answer_safe("What is the capital of France?")
print(f"Out of scope — fallback: {out_of_scope['fallback']}")
print(f"Answer: {out_of_scope['answer']}")
print(f"Top score: {out_of_scope['top_score']:.3f}")

**What just happened?**
- Score thresholding prevents Claude from hallucinating answers when the knowledge base has no relevant content.
- The `fallback` flag lets your application decide whether to show the safe message or escalate to a human.
- **Tune `min_score` on your labeled eval set** — 0.35 is a starting point, not a universal value.
- Log all fallbacks in production — they reveal gaps in your knowledge base coverage.

In [ ]:
# Challenge: Add re-ranking to the RAG pipeline
# Problem: embedding similarity is approximate. A re-ranker checks retrieved chunks
# again with Claude to filter out irrelevant ones before answering.
#
# Implement rerank() that:
# 1. Takes a query and list of retrieved chunks
# 2. Asks Claude to rate each chunk's relevance 1-5
# 3. Returns chunks with rating >= 3, sorted by rating descending

def rerank(query: str, chunks: list[dict]) -> list[dict]:
    # Your solution here
    # Hint: for each chunk, ask Claude: "Rate the relevance of this text to the query 1-5."
    pass


test_query = "How does caching work?"
retrieved = store.search(test_query, top_k=5)
reranked = rerank(test_query, retrieved)

print(f"Before reranking: {len(retrieved)} chunks")
print(f"After reranking: {len(reranked)} chunks")
print("✓ Challenge setup complete — implement rerank() above!")

---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| Chunking | Fixed-size with overlap; chunk size is your #1 RAG hyperparameter |
| Embeddings | Normalized dot product = cosine similarity |
| Score threshold | Below 0.35 → don't answer; prevents hallucination |
| Precision@k | Measure retrieval quality before optimizing the prompt |
| Fallback | Always handle the out-of-scope case explicitly |

> **Tip:** Chunk size is the most overlooked RAG parameter. Test 256, 512, and 1024 tokens — the best size depends entirely on your content.

---
## What's next
**Day 7** → Advanced Claude Features: extended thinking, vision, PDFs, prompt caching, and citations.

Mark Day 6 complete in your [tracker](../index.html).